# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1. Modelos supervisados y métricas (Semana 12)

Profesor: Elias Buitrago Bolivar  
Universidad Central  
Caso: Brazilian E-Commerce Public Dataset by Olist

Objetivo: construir un modelo base para anticipar entregas tardías a partir de datos de compras, productos, clientes, vendedores y pagos.

Las instrucciones de GitHub y entrega se encuentran en el documento técnico. Las secciones de carga y preparación se suministran resueltas; complete las cuatro actividades de modelado y evaluación.

## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

In [2]:
ORIGEN = ('https://raw.githubusercontent.com/spdrio/'
          'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
          'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')

pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')

print('Pedidos:', pedidos.shape)
print('Items:', items.shape)
print('Productos:', productos.shape)
print('Clientes:', clientes.shape)
print('Vendedores:', vendedores.shape)
print('Pagos:', pagos.shape)

Pedidos: (99441, 8)
Items: (112650, 7)
Productos: (32951, 9)
Clientes: (99441, 5)
Vendedores: (3095, 4)
Pagos: (103886, 5)


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [3]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [4]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

Pedidos integrados: 98666


In [5]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

Pedidos disponibles: 96470
Entregas tardías: 6.77 %


### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [6]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

Variables predictoras: 28
Numéricas: 24 | Categóricas: 4


### Muestra individual

Cada estudiante utilizará 35.000 pedidos, seleccionados a partir del ID numérico entregado por el profesor (por ejemplo, ID 01). El mismo ID genera siempre la misma muestra y las mismas particiones.

Se utilizarán 24.500 pedidos para entrenar, 5.250 para validar y 5.250 para la prueba final. Durante las semanas 12 a 15 no se utilizarán los datos de prueba para elegir modelos.

En la siguiente celda cambie `ID_ESTUDIANTE` por el número asignado por el profesor. Por ejemplo, el ID 01 se escribe como `1`.


In [7]:
# Escriba el ID que le asignó el profesor (01 se escribe 1).
ID_ESTUDIANTE = 7
assert isinstance(ID_ESTUDIANTE, int) and 1 <= ID_ESTUDIANTE <= 30, (
    'El ID debe ser un número entero entre 1 y 30.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)

Estudiante ID: 07 | Semilla: 1007
Muestra: 35000 | Train: 24500 | Valid: 5250 | Test: 5250
Huella de muestra: 3d3c38ac538c


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [8]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)

print('Accuracy referencia:', round(accuracy_score(y_valid, pred_ref), 3))
print('F1 referencia:', round(f1_score(y_valid, pred_ref, zero_division=0), 3))

Accuracy referencia: 0.932
F1 referencia: 0.0


### Actividad 1. Entrenamiento

Complete el entrenamiento de la regresión logística usando `preprocesamiento`. Pruebe `class_weight=None` o `'balanced'`, y explique su elección en la sección de interpretación. Conserve las variables `modelo` y `pred`.

En el estimador utilice `class_weight=peso_clases` y `max_iter=1000` como límite de iteraciones. Si aparece una advertencia de convergencia, revise el entrenamiento.


In [9]:
# TODO 
import warnings

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning


def entrenar_con_aviso(nombre, peso):
    clasificador = LogisticRegression(
        class_weight=peso, max_iter=1000, random_state=semilla)
    pipeline = make_pipeline(clone(preprocesamiento), clasificador)

    with warnings.catch_warnings(record=True) as capturadas:
        warnings.simplefilter('always', ConvergenceWarning)
        pipeline.fit(X_train, y_train)

    avisos = [w for w in capturadas
              if issubclass(w.category, ConvergenceWarning)]
    if avisos:
        print(f'ADVERTENCIA DE CONVERGENCIA en {nombre}:')
        for aviso in avisos:
            print(f'  {aviso.message}')
    else:
        print(f'Sin advertencia de convergencia en {nombre}.')
    return pipeline



peso_clases = 'balanced'  
modelo = entrenar_con_aviso('modelo principal', peso_clases)
pred = modelo.predict(X_valid)

valores_pred, conteos_pred = np.unique(pred, return_counts=True)
print('Tamaño de pred:', len(pred))
print('Valores únicos de pred:', valores_pred.tolist())
print('Conteo por clase:', dict(zip(valores_pred.tolist(),
                                   conteos_pred.tolist())))
print(f'Pedidos predichos como tardíos: {int((pred == 1).sum())} de {len(pred)}')


comparacion_modelos = []
for etiqueta, peso_cw in [('None', None), ('balanced', 'balanced')]:
    modelo_cw = entrenar_con_aviso(f'comparación {etiqueta}', peso_cw)
    pred_cw = modelo_cw.predict(X_valid)
    proba_cw = modelo_cw.predict_proba(X_valid)[:, 1]
    tn_cw, fp_cw, fn_cw, tp_cw = confusion_matrix(y_valid, pred_cw).ravel()
    comparacion_modelos.append({
        'class_weight': etiqueta,
        'Precision': precision_score(y_valid, pred_cw, zero_division=0),
        'Recall': recall_score(y_valid, pred_cw, zero_division=0),
        'F1': f1_score(y_valid, pred_cw, zero_division=0),
        'ROC_AUC': roc_auc_score(y_valid, proba_cw),
        'Average_Precision': average_precision_score(y_valid, proba_cw),
        'FP': int(fp_cw),
        'FN': int(fn_cw),
        'Tardios_predichos': int((pred_cw == 1).sum()),
        'Costo_BRL': int(8 * fp_cw + 35 * fn_cw),
    })

tabla_cw = pd.DataFrame(comparacion_modelos).set_index('class_weight')
print()
print('Comparación :')
print(tabla_cw.round(4).to_string())

Sin advertencia de convergencia en modelo principal.
Tamaño de pred: 5250
Valores únicos de pred: [0, 1]
Conteo por clase: {0: 3419, 1: 1831}
Pedidos predichos como tardíos: 1831 de 5250
Sin advertencia de convergencia en comparación None.
Sin advertencia de convergencia en comparación balanced.

Comparación :
              Precision  Recall      F1  ROC_AUC  Average_Precision    FP   FN  Tardios_predichos  Costo_BRL
class_weight                                                                                                
None             1.0000  0.0028  0.0056   0.7164             0.1783     0  355                  1      12425
balanced         0.1223  0.6292  0.2048   0.7127             0.1685  1607  132               1831      17476


## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [15]:
proba = modelo.predict_proba(X_valid)[:, 1]
medidas = {
    'accuracy': float(accuracy_score(y_valid, pred)),
    'precision': float(precision_score(y_valid, pred, zero_division=0)),
    'recall': float(recall_score(y_valid, pred, zero_division=0)),
    'f1': float(f1_score(y_valid, pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_valid, proba)),
    'ap': float(average_precision_score(y_valid, proba)),
}


matriz_val = confusion_matrix(y_valid, pred)
tn_val, fp_val, fn_val, tp_val = matriz_val.ravel()
tabla_matriz_val = pd.DataFrame(
    [[tn_val, fp_val], [fn_val, tp_val]],
    index=['Real 0', 'Real 1'],
    columns=['Pred 0', 'Pred 1'],
)

tabla_medidas = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC',
                'Average Precision'],
    'Valor': [medidas['accuracy'], medidas['precision'], medidas['recall'],
              medidas['f1'], medidas['roc_auc'], medidas['ap']],
})
print('Métricas en validación:')
print(tabla_medidas.round(4).to_string(index=False))
print()
print('Matriz de confusión de validación:')
print(tabla_matriz_val.to_string())
print(f'TN={tn_val}, FP={fp_val}, FN={fn_val}, TP={tp_val}')
print(f'Prevalencia de tardíos en validación: {y_valid.mean():.4f} '
      f'({100 * y_valid.mean():.2f}%)')
print()


ref_pred = referencia.predict(X_valid)
ref_acc = float(accuracy_score(y_valid, ref_pred))
ref_recall = float(recall_score(y_valid, ref_pred, zero_division=0))
ref_f1 = float(f1_score(y_valid, ref_pred, zero_division=0))
tabla_comparacion = pd.DataFrame([
    {'Modelo': 'Regresión logística (balanced)',
     'Accuracy': medidas['accuracy'], 'Recall': medidas['recall'],
     'F1': medidas['f1']},
    {'Modelo': 'Referencia (most_frequent)',
     'Accuracy': ref_acc, 'Recall': ref_recall, 'F1': ref_f1},
])
print('Comparación con el modelo de referencia:')
print(tabla_comparacion.round(4).to_string(index=False))

Métricas en validación:
          Métrica  Valor
         Accuracy 0.6688
        Precision 0.1223
           Recall 0.6292
               F1 0.2048
          ROC-AUC 0.7127
Average Precision 0.1685

Matriz de confusión de validación:
        Pred 0  Pred 1
Real 0    3287    1607
Real 1     132     224
TN=3287, FP=1607, FN=132, TP=224
Prevalencia de tardíos en validación: 0.0678 (6.78%)

Comparación con el modelo de referencia:
                        Modelo  Accuracy  Recall     F1
Regresión logística (balanced)    0.6688  0.6292 0.2048
    Referencia (most_frequent)    0.9322  0.0000 0.0000


### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [16]:
assert pred is not None, 'Complete primero la Actividad 1 (variable pred).'
analisis = valid.copy()
analisis['pred'] = pred
analisis['error'] = (analisis['is_late'] != analisis['pred']).astype(int)


UMBRAL_N = 100


def _wilson(errores, n, z=1.96):
    """Intervalo de Wilson al 95 % para una proporción, en porcentaje."""
    if n == 0:
        return np.nan, np.nan
    p = errores / n
    denominador = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / denominador
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominador
    return 100 * max(0.0, centro - margen), 100 * min(1.0, centro + margen)


def _resumir_por(datos, columna):
    filas = []
    for clave, grupo in datos.groupby(columna, sort=True):
        n = len(grupo)
        errores = int(grupo['error'].sum())
        fp = int(((grupo['pred'] == 1) & (grupo['is_late'] == 0)).sum())
        fn = int(((grupo['pred'] == 0) & (grupo['is_late'] == 1)).sum())
        ic_inf, ic_sup = _wilson(errores, n)
        filas.append({
            columna: clave,
            'n': n,
            'tardios_reales': int((grupo['is_late'] == 1).sum()),
            'fp': fp,
            'fn': fn,
            'pct_error': 100 * errores / n,
            'ic_inf': ic_inf,
            'ic_sup': ic_sup,
        })
    return pd.DataFrame(filas)



tabla_mes = (_resumir_por(analisis, 'mes_compra')
             .set_index('mes_compra')
             .reindex(range(1, 13))
             .reset_index())
if (tabla_mes['n'] < UMBRAL_N).any():
    tabla_mes['muestra_pequena'] = tabla_mes['n'] < UMBRAL_N
else:
    print(f'Ningún mes tiene n < {UMBRAL_N}; no se agrega muestra_pequena.')


conteo_estado = analisis['customer_state'].value_counts()
estados_grandes = conteo_estado[conteo_estado >= UMBRAL_N].index.tolist()
estados_pequenos = sorted(conteo_estado[conteo_estado < UMBRAL_N].index.tolist())
analisis_estado = analisis.copy()
analisis_estado['estado_agrupado'] = analisis_estado['customer_state'].where(
    analisis_estado['customer_state'].isin(estados_grandes), 'Otros estados')
resumen_estado = _resumir_por(analisis_estado, 'estado_agrupado')
filas_estado = (resumen_estado[resumen_estado['estado_agrupado'] != 'Otros estados']
                .sort_values('pct_error', ascending=False))
otros_estado = resumen_estado[resumen_estado['estado_agrupado'] == 'Otros estados']
tabla_estado = pd.concat([filas_estado, otros_estado], ignore_index=True)

formato = {'pct_error': '{:.2f}'.format,
           'ic_inf': '{:.2f}'.format,
           'ic_sup': '{:.2f}'.format}
print()
print('Errores por mes de compra:')
print(tabla_mes.to_string(index=False, formatters=formato))
print()
print(f"Estados agrupados en 'Otros estados' ({len(estados_pequenos)}): "
      f"{estados_pequenos}")
print('Errores por estado del cliente:')
print(tabla_estado.to_string(index=False, formatters=formato))

error_global = 100 * analisis['error'].mean()
print(f'\nPorcentaje de error global de validación: {error_global:.2f}%')


mes_mayor = tabla_mes.loc[tabla_mes['pct_error'].idxmax()]
estado_mayor = filas_estado.loc[filas_estado['pct_error'].idxmax()]
mes_mas_fn = tabla_mes.loc[tabla_mes['fn'].idxmax()]
estado_mas_fn = filas_estado.loc[filas_estado['fn'].idxmax()]

print(f"Mes con mayor pct_error: {int(mes_mayor['mes_compra'])} "
      f"({mes_mayor['pct_error']:.2f}%).")
print(f"Estado individual con mayor pct_error: {estado_mayor['estado_agrupado']} "
      f"({estado_mayor['pct_error']:.2f}%).")
print(f"Mes con más falsos negativos: {int(mes_mas_fn['mes_compra'])} "
      f"({int(mes_mas_fn['fn'])} FN).")
print(f"Estado individual con más falsos negativos: {estado_mas_fn['estado_agrupado']} "
      f"({int(estado_mas_fn['fn'])} FN).")

mes_ci_no_incluye = not (mes_mayor['ic_inf'] <= error_global <= mes_mayor['ic_sup'])
estado_ci_no_incluye = not (
    estado_mayor['ic_inf'] <= error_global <= estado_mayor['ic_sup'])
print(f"IC95% del mes con mayor error: [{mes_mayor['ic_inf']:.2f}, "
      f"{mes_mayor['ic_sup']:.2f}] -> "
      f"{'NO incluye' if mes_ci_no_incluye else 'incluye'} el global.")
print(f"IC95% del estado con mayor error: [{estado_mayor['ic_inf']:.2f}, "
      f"{estado_mayor['ic_sup']:.2f}] -> "
      f"{'NO incluye' if estado_ci_no_incluye else 'incluye'} el global.")

Ningún mes tiene n < 100; no se agrega muestra_pequena.

Errores por mes de compra:
 mes_compra   n  tardios_reales  fp  fn pct_error ic_inf ic_sup
          1 456              15 128   5     29.17  25.18  33.50
          2 437              65 113  21     30.66  26.52  35.14
          3 498              72 167  19     37.35  33.21  41.68
          4 498              32 151  10     32.33  28.37  36.56
          5 558              33 179  15     34.77  30.93  38.81
          6 510               8 112   7     23.33  19.87  27.20
          7 538              11 210   2     39.41  35.37  43.60
          8 559              27 269   7     49.37  45.25  53.51
          9 249              15  80   6     34.54  28.91  40.64
         10 246              11  60   7     27.24  22.05  33.12
         11 401              44  98  18     28.93  24.71  33.55
         12 300              23  40  15     18.33  14.36  23.10

Estados agrupados en 'Otros estados' (17): ['AC', 'AL', 'AM', 'AP', 'CE', 'MA', 'MS

### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

In [12]:
# Actividad 4 resuelta: costo total de los errores en validación.
# Supuesto ilustrativo del enunciado, no costos verificados de Olist.
COSTO_FP = 8
COSTO_FN = 35

costo_tn, costo_fp, costo_fn, costo_tp = confusion_matrix(y_valid, pred).ravel()
costo_fp_fn = int(COSTO_FP * costo_fp + COSTO_FN * costo_fn)

costo_parte_fp = int(COSTO_FP * costo_fp)
costo_parte_fn = int(COSTO_FN * costo_fn)
print(f'Costo total de los errores en validación: R$ {costo_fp_fn}')
print('Desglose:')
print(f'  FP={int(costo_fp)} x R$ {COSTO_FP} = R$ {costo_parte_fp} '
      f'({100 * costo_parte_fp / costo_fp_fn:.2f}% del total)')
print(f'  FN={int(costo_fn)} x R$ {COSTO_FN} = R$ {costo_parte_fn} '
      f'({100 * costo_parte_fn / costo_fp_fn:.2f}% del total)')
print()

# Costos informativos de referencia y de class_weight=None (sin reentrenar).
costo_ref_tn, costo_ref_fp, costo_ref_fn, costo_ref_tp = confusion_matrix(
    y_valid, referencia.predict(X_valid)).ravel()
costo_ref_total = int(COSTO_FP * costo_ref_fp + COSTO_FN * costo_ref_fn)
costo_none_fp = int(tabla_cw.loc['None', 'FP'])
costo_none_fn = int(tabla_cw.loc['None', 'FN'])
costo_none_total = int(tabla_cw.loc['None', 'Costo_BRL'])
print('Costos informativos en validación:')
print(f'  Referencia (most_frequent): FP={int(costo_ref_fp)}, '
      f'FN={int(costo_ref_fn)}, costo=R$ {costo_ref_total}')
print(f'  Regresión logística (class_weight=None): FP={costo_none_fp}, '
      f'FN={costo_none_fn}, costo=R$ {costo_none_total}')
print()

# Exploración informativa por umbral: no cambia el modelo ni pred.
costo_umbrales = [0.5, 0.4, 0.3, 0.2, 0.19]
costo_filas_umbral = []
for costo_umbral in costo_umbrales:
    costo_pred_umbral = (proba >= costo_umbral).astype(int)
    costo_umbral_tn, costo_umbral_fp, costo_umbral_fn, costo_umbral_tp = (
        confusion_matrix(y_valid, costo_pred_umbral).ravel())
    costo_filas_umbral.append({
        'umbral': costo_umbral,
        'FP': int(costo_umbral_fp),
        'FN': int(costo_umbral_fn),
        'tardios_predichos': int(costo_pred_umbral.sum()),
        'Recall': float(recall_score(y_valid, costo_pred_umbral,
                                     zero_division=0)),
        'Precision': float(precision_score(y_valid, costo_pred_umbral,
                                           zero_division=0)),
        'Costo_BRL': int(COSTO_FP * costo_umbral_fp +
                         COSTO_FN * costo_umbral_fn),
    })
costo_tabla_umbral = pd.DataFrame(costo_filas_umbral)
print('Exploración informativa por umbral (no cambia el modelo ni pred):')
print(costo_tabla_umbral.round(4).to_string(index=False))

### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [13]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


AssertionError: Complete las métricas de la Actividad 2.

## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [ ]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

## 6. Interpretación de negocio

1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.

Respuesta: En la validación de 5.250 pedidos, de los cuales 356 llegaron tarde, comparé el modelo de referencia con la regresión logística. El modelo de referencia obtuvo una Accuracy de 93,2 %, es decir, acertó en la mayoría de los pedidos, pero no identificó ninguno de los que llegaron tarde. Por eso, su Recall y F1 fueron 0. En cambio, la regresión logística con class_weight='balanced' tuvo una Accuracy de 66,9 %, pero logró identificar 224 de los 356 pedidos tardíos. Esto representa un Recall de 62,9 %. El F1 es bajo (0,205) porque la Precision es de apenas 12,2 %. Elegí esta configuración porque es más útil para detectar pedidos que podrían retrasarse. Con class_weight=None, el modelo solo identificó un pedido tardío. Aunque el modelo 'balanced' detecta más retrasos, también marca como tardíos muchos pedidos que finalmente llegan a tiempo. Esto genera falsas alarmas y aumenta el costo a R$ 17.476, frente a R$ 12.425

2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?

Respuesta:

3. Proponga una mejora concreta para la semana 13. Considere la información del producto, del vendedor, del pago o el tratamiento de variables categóricas. Explique cómo verificará la mejora usando la misma partición de validación.

Respuesta:

## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [ ]:
import sys
import sklearn

reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

ruta = Path(f'reports/S12_resultados_ID_{identificador}.json')
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reporte:', ruta)

# En Colab, descarga el JSON para conservarlo fuera de la sesión.
try:
    from google.colab import files
    files.download(str(ruta))
except Exception:
    print('Si no se descargó automáticamente, guarde manualmente:', ruta)
